# 04 — Build BM25 Index (Tantivy)

Build one global disk-backed sparse index for comments.

The implementation lives in `src/rag/retrieval/bm25.py`. Tantivy provides a
real inverted index with bounded indexing memory and global BM25 statistics.

In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.preprocessing.processor import TextProcessor
from src.rag.retrieval.bm25 import BM25Retriever

COMMENTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "comments_clean.parquet"
)

INDEX_DIR = (
    PROJECT_ROOT
    / "data"
    / "indexes"
    / "product_comments_bm25_tantivy"
)

BATCH_SIZE = 50_000
WRITER_HEAP_SIZE = 128_000_000
NUM_THREADS = 1
COMMIT_EVERY_BATCHES = 10
OVERWRITE = True

## Build index

In [ ]:
processor = TextProcessor()

start = time.perf_counter()

manifest = BM25Retriever.build_from_parquet(
    input_path=COMMENTS_PATH,
    output_path=INDEX_DIR,
    processor=processor,
    batch_size=BATCH_SIZE,
    writer_heap_size=WRITER_HEAP_SIZE,
    num_threads=NUM_THREADS,
    commit_every_batches=COMMIT_EVERY_BATCHES,
    overwrite=OVERWRITE,
)

print("Elapsed minutes:", round((time.perf_counter() - start) / 60, 2))
print(manifest)

## Smoke test

In [ ]:
bm25 = BM25Retriever(processor=processor)
bm25.load(INDEX_DIR)

query = "ضد آفتاب پوست چرب جوش"

start = time.perf_counter()
results = bm25.retrieve(query, top_k=5)
latency_ms = (time.perf_counter() - start) * 1000

print("Latency (ms):", round(latency_ms, 2))
display(results[["id", "product_id", "score", "body"]])